# Central limma correction and all-modality k-means inputs

Each omics layer is corrected separately with `limma::removeBatchEffect`, preserving the four Quartet donors in the design. The corrected layers are then transformed into equal-weight modality blocks for a simple all-three-modality k-means analysis.

In [ ]:
library(tidyverse)
library(limma)

WORKDIR <- if (dir.exists("figshare_data")) "." else "evaluation_data/multiomics"
BEFORE_DIR <- file.path(WORKDIR, "before")
AFTER_DIR <- file.path(WORKDIR, "after")
PLOTS_DIR <- file.path(WORKDIR, "plots")

dir.create(AFTER_DIR, showWarnings = FALSE, recursive = TRUE)
dir.create(PLOTS_DIR, showWarnings = FALSE, recursive = TRUE)

MODALITIES <- c("Transcriptomics", "Proteomics", "Metabolomics")
DONOR_LEVELS <- c("D5", "D6", "F7", "M8")
TOP_PCA_FEATURES <- 3000
BOXPLOT_MAX_FEATURES <- 5000
K_CLUSTERS <- 4
KMEANS_SEED <- 11

## Helper functions

Diagnostics use top-variable features for PCA and a capped feature set for distribution plots so transcriptomics remains responsive.

In [ ]:
read_feature_matrix <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)
  first_col <- names(df)[1]
  mat <- df %>%
    column_to_rownames(first_col) %>%
    as.data.frame(check.names = FALSE)
  mat[] <- lapply(mat, as.numeric)
  as.matrix(mat)
}

write_feature_matrix <- function(expr, path) {
  out <- as.data.frame(expr, check.names = FALSE) %>% rownames_to_column("rowname")
  write.table(out, file = path, sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)
}

load_prepared_modality <- function(modality) {
  modality_dir <- file.path(BEFORE_DIR, modality)
  expr <- read_feature_matrix(file.path(modality_dir, "central_intensities_log_UNION.tsv"))
  metadata <- readr::read_tsv(file.path(modality_dir, "metadata.tsv"), show_col_types = FALSE) %>%
    mutate(
      condition = factor(condition, levels = DONOR_LEVELS),
      batch = factor(batch),
      rep = as.integer(rep)
    )

  missing <- setdiff(metadata$file, colnames(expr))
  if (length(missing) > 0) stop(modality, ": missing matrix samples: ", paste(head(missing), collapse = ", "), call. = FALSE)
  expr <- expr[, metadata$file, drop = FALSE]
  list(expr = expr, metadata = metadata)
}

assert_limma_design <- function(metadata, modality) {
  if (nrow(metadata) != 180) stop(modality, ": expected 180 samples.", call. = FALSE)
  if (n_distinct(metadata$batch) != 15) stop(modality, ": expected 15 batches.", call. = FALSE)
  if (any(metadata %>% count(condition) %>% pull(n) != 45)) stop(modality, ": expected 45 samples per donor.", call. = FALSE)
  if (any(metadata %>% count(batch_code, batch, condition) %>% pull(n) != 3)) {
    stop(modality, ": expected 3 donor replicates in each batch.", call. = FALSE)
  }

  full_design <- model.matrix(~ condition + batch, data = metadata)
  rank <- qr(full_design)$rank
  if (rank != ncol(full_design)) {
    stop(modality, ": limma design is not full rank: ", rank, " < ", ncol(full_design), call. = FALSE)
  }
  invisible(TRUE)
}

prepare_pca <- function(expr, metadata, top_n = TOP_PCA_FEATURES) {
  feature_var <- apply(expr, 1, var, na.rm = TRUE)
  keep <- which(is.finite(feature_var) & feature_var > 0)
  keep <- keep[order(feature_var[keep], decreasing = TRUE)]
  keep <- head(keep, min(top_n, length(keep)))
  if (length(keep) < 2) stop("Need at least two variable features for PCA.", call. = FALSE)

  x <- t(expr[keep, , drop = FALSE])
  if (anyNA(x)) {
    for (j in seq_len(ncol(x))) {
      med <- median(x[, j], na.rm = TRUE)
      if (!is.finite(med)) med <- 0
      x[is.na(x[, j]), j] <- med
    }
  }

  pca <- prcomp(x, center = TRUE, scale. = TRUE)
  variance <- pca$sdev^2 / sum(pca$sdev^2)
  scores <- as_tibble(pca$x[, 1:2], rownames = "file") %>% left_join(metadata, by = "file")
  list(scores = scores, variance = variance, n_features = length(keep))
}

plot_diagnostics <- function(expr, metadata, modality, state_label) {
  pca_res <- prepare_pca(expr, metadata)

  pca_batch <- ggplot(pca_res$scores, aes(PC1, PC2, color = batch_code, shape = condition)) +
    geom_point(size = 2.5, alpha = 0.9) +
    theme_classic() +
    theme(legend.text = element_text(size = 7), legend.title = element_text(size = 9)) +
    labs(
      title = paste(modality, state_label, "by batch"),
      x = paste0("PC1 [", round(pca_res$variance[1] * 100, 1), "%]"),
      y = paste0("PC2 [", round(pca_res$variance[2] * 100, 1), "%]")
    )

  pca_condition <- ggplot(pca_res$scores, aes(PC1, PC2, color = condition)) +
    geom_point(size = 2.5, alpha = 0.9) +
    theme_classic() +
    labs(
      title = paste(modality, state_label, "by donor"),
      x = paste0("PC1 [", round(pca_res$variance[1] * 100, 1), "%]"),
      y = paste0("PC2 [", round(pca_res$variance[2] * 100, 1), "%]")
    )

  set.seed(1)
  plot_features <- if (nrow(expr) > BOXPLOT_MAX_FEATURES) sample(rownames(expr), BOXPLOT_MAX_FEATURES) else rownames(expr)
  long_expr <- expr[plot_features, , drop = FALSE] %>%
    as.data.frame(check.names = FALSE) %>%
    rownames_to_column("feature") %>%
    pivot_longer(-feature, names_to = "file", values_to = "value") %>%
    left_join(metadata, by = "file")

  box_by_batch <- ggplot(long_expr, aes(x = batch_code, y = value, fill = batch_code)) +
    geom_boxplot(outlier.shape = NA) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1), legend.position = "none") +
    labs(title = paste(modality, state_label, "batch distributions"), x = "batch", y = "log-scale value")

  density_by_batch <- ggplot(long_expr, aes(x = value, color = batch_code)) +
    geom_density(na.rm = TRUE) +
    theme_minimal() +
    theme(legend.text = element_text(size = 7), legend.title = element_text(size = 9)) +
    labs(title = paste(modality, state_label, "density"), x = "log-scale value", y = "density")

  prefix <- file.path(PLOTS_DIR, paste0(modality, "_limma_", state_label))
  ggsave(paste0(prefix, "_pca_by_batch.png"), pca_batch, width = 9.5, height = 6.5, bg = "white")
  ggsave(paste0(prefix, "_pca_by_condition.png"), pca_condition, width = 6, height = 5, bg = "white")
  ggsave(paste0(prefix, "_boxplot_by_batch.png"), box_by_batch, width = 9.5, height = 5.5, bg = "white")
  ggsave(paste0(prefix, "_density_by_batch.png"), density_by_batch, width = 9.5, height = 6, bg = "white")

  tibble(modality = modality, state = state_label, pca_features = pca_res$n_features)
}

row_zscore <- function(expr) {
  values <- as.matrix(expr)
  means <- rowMeans(values, na.rm = TRUE)
  sds <- apply(values, 1, sd, na.rm = TRUE)
  sds[!is.finite(sds) | sds == 0] <- 1
  sweep(sweep(values, 1, means, "-"), 1, sds, "/")
}

adjusted_rand_index <- function(true_labels, predicted_labels) {
  tab <- table(true_labels, predicted_labels)
  n <- sum(tab)
  if (n < 2) return(NA_real_)
  choose2 <- function(x) x * (x - 1) / 2
  index <- sum(choose2(tab))
  row_sum <- sum(choose2(rowSums(tab)))
  col_sum <- sum(choose2(colSums(tab)))
  total <- choose2(n)
  expected <- row_sum * col_sum / total
  maximum <- (row_sum + col_sum) / 2
  if (maximum == expected) return(0)
  as.numeric((index - expected) / (maximum - expected))
}

## Correct each modality independently

`batch` is the technical factor removed by limma. Donor condition stays in the design so D5/D6/F7/M8 signal is preserved for clustering.

In [ ]:
before_matrices <- list()
corrected_matrices <- list()
metadata_by_modality <- list()
diagnostic_summaries <- list()

correction_summaries <- purrr::map_dfr(MODALITIES, function(modality) {
  message("Correcting ", modality)
  prepared <- load_prepared_modality(modality)
  expr <- prepared$expr
  metadata <- prepared$metadata
  assert_limma_design(metadata, modality)

  diagnostic_summaries[[paste(modality, "before", sep = "_")]] <<- plot_diagnostics(expr, metadata, modality, "before")

  design <- model.matrix(~ condition, data = metadata)
  colnames(design) <- c("Intercept", "D6", "F7", "M8")
  corrected <- limma::removeBatchEffect(as.matrix(expr), batch = metadata$batch, design = design) %>%
    as.data.frame(check.names = FALSE)

  if (anyNA(corrected)) stop(modality, ": corrected matrix contains NA values.", call. = FALSE)
  if (any(!is.finite(as.matrix(corrected)))) stop(modality, ": corrected matrix contains non-finite values.", call. = FALSE)

  modality_after_dir <- file.path(AFTER_DIR, modality)
  dir.create(modality_after_dir, showWarnings = FALSE, recursive = TRUE)
  write_feature_matrix(corrected, file.path(modality_after_dir, "intensities_log_Rcorrected_UNION.tsv"))
  write.table(metadata, file = file.path(modality_after_dir, "metadata.tsv"),
              sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

  diagnostic_summaries[[paste(modality, "after", sep = "_")]] <<- plot_diagnostics(as.matrix(corrected), metadata, modality, "after")

  before_matrices[[modality]] <<- expr
  corrected_matrices[[modality]] <<- as.matrix(corrected)
  metadata_by_modality[[modality]] <<- metadata

  tibble(
    modality = modality,
    features_before = nrow(expr),
    features_after = nrow(corrected),
    samples = ncol(corrected),
    batches = n_distinct(metadata$batch),
    design_rank = qr(model.matrix(~ condition + batch, data = metadata))$rank,
    design_columns = ncol(model.matrix(~ condition + batch, data = metadata))
  )
})

correction_summaries

## Combine modality blocks for k-means

There is no true shared library ID across omics layers. Samples are pseudo-matched by deterministic batch rank, donor, and replicate, then each modality block is row z-scored and divided by `sqrt(number_of_features)` so no modality dominates by feature count.

In [ ]:
sample_keys <- sort(unique(metadata_by_modality[[MODALITIES[1]]]$pseudo_sample))

for (modality in MODALITIES) {
  keys <- sort(unique(metadata_by_modality[[modality]]$pseudo_sample))
  if (!identical(keys, sample_keys)) stop(modality, ": pseudo-sample keys do not match across modalities.", call. = FALSE)
}

build_modality_block <- function(expr, metadata, modality, sample_keys) {
  mat <- expr[, metadata$file, drop = FALSE]
  colnames(mat) <- metadata$pseudo_sample
  if (anyDuplicated(colnames(mat)) > 0) stop(modality, ": duplicate pseudo-sample columns.", call. = FALSE)
  mat <- mat[, sample_keys, drop = FALSE]
  scaled <- row_zscore(mat) / sqrt(nrow(mat))
  rownames(scaled) <- paste0(modality, "__", make.unique(rownames(scaled)))
  scaled
}

before_blocks <- purrr::map(MODALITIES, ~ build_modality_block(before_matrices[[.x]], metadata_by_modality[[.x]], .x, sample_keys))
corrected_blocks <- purrr::map(MODALITIES, ~ build_modality_block(corrected_matrices[[.x]], metadata_by_modality[[.x]], .x, sample_keys))

names(before_blocks) <- MODALITIES
names(corrected_blocks) <- MODALITIES

before_kmeans_matrix <- do.call(rbind, before_blocks)
corrected_kmeans_matrix <- do.call(rbind, corrected_blocks)

if (anyNA(before_kmeans_matrix) || anyNA(corrected_kmeans_matrix)) stop("Combined k-means matrices contain NA values.", call. = FALSE)
if (any(!is.finite(before_kmeans_matrix)) || any(!is.finite(corrected_kmeans_matrix))) stop("Combined k-means matrices contain non-finite values.", call. = FALSE)
if (anyDuplicated(rownames(before_kmeans_matrix)) > 0 || anyDuplicated(rownames(corrected_kmeans_matrix)) > 0) stop("Duplicate combined feature IDs.", call. = FALSE)

combined_metadata <- metadata_by_modality[[MODALITIES[1]]] %>%
  select(pseudo_sample, batch_code, condition, rep) %>%
  distinct()

for (modality in MODALITIES) {
  modality_meta <- metadata_by_modality[[modality]] %>%
    select(pseudo_sample, file, batch, lab, platform, protocol, date) %>%
    rename_with(~ paste0(modality, "_", .x), -pseudo_sample)
  combined_metadata <- combined_metadata %>% left_join(modality_meta, by = "pseudo_sample")
}

combined_metadata <- combined_metadata %>%
  mutate(pseudo_sample = factor(pseudo_sample, levels = sample_keys)) %>%
  arrange(pseudo_sample) %>%
  mutate(pseudo_sample = as.character(pseudo_sample))

stopifnot(identical(colnames(before_kmeans_matrix), combined_metadata$pseudo_sample))
stopifnot(identical(colnames(corrected_kmeans_matrix), combined_metadata$pseudo_sample))

write_feature_matrix(before_kmeans_matrix, file.path(AFTER_DIR, "all_modalities_before_kmeans_matrix.tsv"))
write_feature_matrix(corrected_kmeans_matrix, file.path(AFTER_DIR, "all_modalities_corrected_kmeans_matrix.tsv"))
write.table(combined_metadata, file = file.path(AFTER_DIR, "all_modalities_metadata.tsv"),
            sep = "\t", quote = TRUE, row.names = FALSE, col.names = TRUE)

tibble(
  matrix = c("before", "corrected"),
  features = c(nrow(before_kmeans_matrix), nrow(corrected_kmeans_matrix)),
  samples = c(ncol(before_kmeans_matrix), ncol(corrected_kmeans_matrix))
)

## Central k-means sanity check

The clustering result is only a lightweight check that the combined matrix is usable. Donor ARI measures biological recovery; batch ARI measures remaining technical structure.

In [ ]:
run_kmeans <- function(feature_by_sample, k = K_CLUSTERS, seed = KMEANS_SEED) {
  set.seed(seed)
  km <- kmeans(t(feature_by_sample), centers = k, nstart = 50, iter.max = 100)
  as.integer(km$cluster)
}

before_clusters <- run_kmeans(before_kmeans_matrix)
corrected_clusters <- run_kmeans(corrected_kmeans_matrix)

cluster_assignments <- combined_metadata %>%
  transmute(
    pseudo_sample,
    condition = as.character(condition),
    batch_code,
    Before_CtrlKm_4clusters = before_clusters,
    Cor_CtrlKm_4clusters = corrected_clusters
  )

kmeans_metrics <- tibble(
  matrix = c("before", "before", "corrected", "corrected"),
  target = c("donor", "batch", "donor", "batch"),
  k = K_CLUSTERS,
  ARI = c(
    adjusted_rand_index(cluster_assignments$condition, cluster_assignments$Before_CtrlKm_4clusters),
    adjusted_rand_index(cluster_assignments$batch_code, cluster_assignments$Before_CtrlKm_4clusters),
    adjusted_rand_index(cluster_assignments$condition, cluster_assignments$Cor_CtrlKm_4clusters),
    adjusted_rand_index(cluster_assignments$batch_code, cluster_assignments$Cor_CtrlKm_4clusters)
  ),
  N = nrow(cluster_assignments)
)

write.table(cluster_assignments, file = file.path(AFTER_DIR, "all_modalities_kmeans_clusters.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
write.table(kmeans_metrics, file = file.path(AFTER_DIR, "all_modalities_kmeans_metrics.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

kmeans_metrics

## Save summaries

In [ ]:
diagnostic_summaries_df <- bind_rows(diagnostic_summaries)

write.table(correction_summaries, file = file.path(AFTER_DIR, "central_correction_summary.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
write.table(diagnostic_summaries_df, file = file.path(AFTER_DIR, "diagnostic_plot_summary.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)

correction_summaries
diagnostic_summaries_df

## Session information

In [ ]:
sessionInfo()